In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# ----------------------------------------------------
prompt      = "Die Bayerische Staatsoper ist"
model_id    = "dbmdz/german-gpt2"           # z. B. "dbmdz/german-gpt2" für Deutsch
top_k       = 10                # Anzahl der anzuzeigenden Kandidaten
temperature = 0.6              # <1 deterministischer, >1 zufälliger
# ----------------------------------------------------

# Modell & Tokenizer laden
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(model_id)
model.eval()

# Prompt tokenisieren
input_ids = tokenizer(prompt, return_tensors="pt").input_ids  # shape (1, seq_len)

# Vorwärtsdurchlauf (ohne Gradienten)
with torch.no_grad():
    logits = model(input_ids).logits                 # shape (1, seq_len, vocab_size)

# Logits des letzten Tokens
next_token_logits = logits[0, -1, :]                 # shape (vocab_size,)

# Temperaturskalierung und Wahrscheinlichkeiten
scaled_logits = next_token_logits / temperature
probs         = torch.softmax(scaled_logits, dim=-1)

# Top-k wählen
values, indices = torch.topk(probs, top_k)

# Ausgabe
print(f"Top-{top_k}-Kandidaten (Temperatur = {temperature}) nach:\n»{prompt}«\n")
for rank, (idx, p) in enumerate(zip(indices.tolist(), values.tolist()), start=1):
    token = tokenizer.decode([idx]).strip()
    print(f"{rank}. '{token}'  –  p = {p:.4f}")

# Optional: Prompt mit sampling & Temperatur fortsetzen (1 Token)
generated_ids = model.generate(
    input_ids,
    max_new_tokens=1,
    do_sample=True,            # Sampling aktivieren
    temperature=temperature,   # selbe Temperatur
    top_k=0                    # keine Top-k-Begrenzung beim Generate (optional)
)
print("\nVervollständigte Ausgabe (Sampling):")
print(tokenizer.decode(generated_ids[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Top-10-Kandidaten (Temperatur = 0.6) nach:
»Die Bayerische Staatsoper ist«

1. 'eine'  –  p = 0.3913
2. 'die'  –  p = 0.1653
3. 'ein'  –  p = 0.1565
4. 'das'  –  p = 0.0692
5. 'mit'  –  p = 0.0448
6. 'seit'  –  p = 0.0354
7. 'in'  –  p = 0.0316
8. 'der'  –  p = 0.0156
9. 'für'  –  p = 0.0148
10. 'nicht'  –  p = 0.0073

Vervollständigte Ausgabe (Sampling):
Die Bayerische Staatsoper ist in
